# Smart Manufacturing Intelligence Platform (SMIP)

#  Bronze Layer

## Notebook 01 – Master Data Ingestion

---

## Objective

This notebook ingests all manufacturing master datasets from the Databricks Volume into the Bronze layer of the Medallion Architecture.

The notebook performs the following tasks:

- Reads master data CSV files
- Validates source files
- Adds audit metadata
- Writes Delta tables
- Performs quality checks
- Produces an ingestion summary

---

## Source

Catalog

`smip`

Schema

`bronze`

Volume

`source_data/master_data`

---

## Target

Bronze Delta Tables

- bronze.products
- bronze.machines
- bronze.operators
- bronze.operations
- bronze.press_programs
- bronze.production_halls
- bronze.production_lines
- bronze.stations
- bronze.test_programs
- bronze.tools

---

Author

Sumanth Vempalle

Project

Smart Manufacturing Intelligence Platform

Version

1.1.0

In [0]:
%python
# COMMAND ----------
# Imports

from datetime import datetime

from pyspark.sql import DataFrame

from pyspark.sql.functions import (
    current_timestamp,
    current_date,
    input_file_name,
    lit,
)

print("=" * 65)
print("SMIP - Bronze Master Data Ingestion")
print("=" * 65)

In [0]:
%python
# COMMAND ----------
# Configuration

CATALOG = "smip"

SCHEMA = "bronze"

VOLUME = "source_data"

MASTER_FOLDER = "master_data"

BASE_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{MASTER_FOLDER}"
)

print(f"Catalog      : {CATALOG}")
print(f"Schema       : {SCHEMA}")
print(f"Volume       : {VOLUME}")
print(f"Source Folder: {MASTER_FOLDER}")
print(f"Base Path    : {BASE_PATH}")

In [0]:
%python
# COMMAND ----------
# Master datasets

MASTER_DATASETS = {

    "products": "products.csv",

    "machines": "machines.csv",

    "operators": "operators.csv",

    "operations": "operations.csv",

    "press_programs": "press_programs.csv",

    "production_halls": "production_halls.csv",

    "production_lines": "production_lines.csv",

    "stations": "stations.csv",

    "test_programs": "test_programs.csv",

    "tools": "tools.csv",
}

print(f"Datasets to ingest : {len(MASTER_DATASETS)}")

for table, file in MASTER_DATASETS.items():

    print(f"{table:<22} -> {file}")

In [0]:
%python
# COMMAND ----------
# Verify source folder

display(
    dbutils.fs.ls(BASE_PATH)
)

### Logging Function

In [0]:
%python
# COMMAND ----------
# Logging Utility

def log(message: str) -> None:

    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    print(
        f"[{timestamp}] {message}"
    )

In [0]:
%python
# COMMAND ----------

log("Bronze ingestion started.")

### Read CSV Function

In [0]:
%python
# COMMAND ----------
# Read CSV

def read_csv(file_name: str) -> DataFrame:

    path = f"{BASE_PATH}/{file_name}"

    log(f"Reading {file_name}")

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

    return df

### Display Dataset Information

In [0]:
%python
# COMMAND ----------
# Dataset Information

def dataset_information(
    df: DataFrame,
    table_name: str,
) -> None:

    log(f"Dataset : {table_name}")

    print(f"Rows    : {df.count()}")

    print(f"Columns : {len(df.columns)}")

    print("Schema")

    df.printSchema()

    print("-" * 60)

### Validate Dataset

In [0]:
%python
# COMMAND ----------
# Validation

from pyspark.sql.functions import count

def validate_dataframe(
    df: DataFrame,
    table_name: str,
) -> None:

    row_count = df.count()

    if row_count == 0:

        raise ValueError(
            f"{table_name} is empty."
        )

    duplicate_rows = (

        row_count

        -

        df.dropDuplicates().count()

    )

    log(f"Rows            : {row_count}")

    log(f"Duplicate Rows  : {duplicate_rows}")

    log("Validation Successful")

### Add Audit Columns

In [0]:
%python
# COMMAND ----------
# Audit Columns

def add_audit_columns(
    df: DataFrame,
    source_file: str,
) -> DataFrame:

    return (

        df

        .withColumn(
            "ingestion_timestamp",
            current_timestamp(),
        )

        .withColumn(
            "load_date",
            current_date(),
        )

        .withColumn(
            "source_file",
            lit(source_file),
        )

    )

### Test Everything

In [0]:
%python
# COMMAND ----------

products = read_csv(
    "products.csv"
)

dataset_information(
    products,
    "products",
)

validate_dataframe(
    products,
    "products",
)

products = add_audit_columns(
    products,
    "products.csv",
)

display(products.limit(10))

### Generic Delta Writer

In [0]:
%python
# COMMAND ----------
# Delta Table Writer

def write_delta_table(
    df: DataFrame,
    table_name: str,
) -> None:

    full_table_name = (
        f"{CATALOG}.{SCHEMA}.{table_name}"
    )

    log(
        f"Writing Delta Table : {full_table_name}"
    )

    (

        df.write

        .format("delta")

        .mode("overwrite")

        .option(
            "overwriteSchema",
            "true",
        )

        .saveAsTable(
            full_table_name
        )

    )

    log(
        "Delta table created successfully."
    )

### Verify Table Creation

In [0]:
%python
# COMMAND ----------
# Verify Delta Table

def verify_delta_table(
    table_name: str,
) -> None:

    full_table_name = (
        f"{CATALOG}.{SCHEMA}.{table_name}"
    )

    df = spark.table(
        full_table_name
    )

    log(
        f"Verification : {table_name}"
    )

    print(
        f"Rows : {df.count()}"
    )

    print(
        f"Columns : {len(df.columns)}"
    )

    print("-" * 60)

### Display Table Information

In [0]:
%python
# COMMAND ----------
# Table Information

def table_information(table_name: str) -> None:

    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"

    log(f"Table: {full_table}")

    print("=" * 70)

    print("Columns")

    for column in spark.table(full_table).columns:
        print(f"• {column}")

    print("=" * 70)

    print(f"Total Rows : {spark.table(full_table).count()}")

    print("=" * 70)

    display(
        spark.table(full_table).limit(10)
    )

### Test Delta Write

In [0]:
%python
# COMMAND ----------

products = read_csv(
    "products.csv"
)

validate_dataframe(
    products,
    "products",
)

products = add_audit_columns(
    products,
    "products.csv",
)

write_delta_table(
    products,
    "products",
)

verify_delta_table(
    "products"
)

table_information(
    "products"
)

display(

    spark.table(
        "smip.bronze.products"
    )

)

In [0]:
%python
display(
    spark.sql("SHOW TABLES IN smip.bronze")
)

In [0]:
%python
spark.table("smip.bronze.products").show(5)

### Ingestion Engine


In [0]:
%python
# COMMAND ----------
# Bronze Master Data Ingestion

ingestion_results = []

total_rows_loaded = 0

log("Starting Bronze Master Data Ingestion")

print("=" * 80)

for table_name, file_name in MASTER_DATASETS.items():

    try:

        log(f"Processing {file_name}")

        # Read CSV
        df = read_csv(file_name)

        # Validate
        validate_dataframe(df, table_name)

        # Add audit columns
        df = add_audit_columns(
            df,
            file_name,
        )

        row_count = df.count()

        # Write Delta
        write_delta_table(
            df,
            table_name,
        )

        total_rows_loaded += row_count

        ingestion_results.append({

            "Dataset": table_name,

            "Rows": row_count,

            "Status": "SUCCESS"

        })

        log(f"{table_name} completed successfully.")

        print("-" * 80)

    except Exception as e:

        ingestion_results.append({

            "Dataset": table_name,

            "Rows": 0,

            "Status": "FAILED"

        })

        print(f"ERROR : {table_name}")

        print(str(e))

        print("-" * 80)

### Execution Summary

In [0]:
%python
# COMMAND ----------

summary_df = spark.createDataFrame(
    ingestion_results
)

display(summary_df)

### Overall Summary

In [0]:
%python
# COMMAND ----------

successful_tables = len(

    [x for x in ingestion_results
     if x["Status"] == "SUCCESS"]

)

failed_tables = len(

    [x for x in ingestion_results
     if x["Status"] == "FAILED"]

)

print("=" * 80)

print("SMIP BRONZE MASTER DATA INGESTION")

print("=" * 80)

print(f"Datasets Processed : {len(MASTER_DATASETS)}")

print(f"Successful Tables : {successful_tables}")

print(f"Failed Tables     : {failed_tables}")

print(f"Rows Loaded       : {total_rows_loaded:,}")

print("=" * 80)

### Verify All Bronze Tables

In [0]:
%python
# COMMAND ----------

display(

    spark.sql(

        """
        SHOW TABLES IN smip.bronze
        """

    )

)


### Preview All Tables

In [0]:
%python
# COMMAND ----------

for table_name in MASTER_DATASETS.keys():

    print("=" * 80)

    print(table_name.upper())

    print("=" * 80)

    display(

        spark.table(

            f"smip.bronze.{table_name}"

        ).limit(5)

    )